# SDS Benchmark — Generation, OCR & NER

End-to-end notebook: generate synthetic documents, run OCR, run NER, view results.

**Setup** (run once):
```
pip install faker jinja2 playwright PyMuPDF rdflib pyyaml Pillow pandas numpy
playwright install chromium
```

GPU models also need: `pip install torch transformers vllm`

## 1. Configuration

In [ ]:
import sys
from pathlib import Path

PROJECT = Path(".").resolve()
if not (PROJECT / "synthdoc").exists():
    PROJECT = PROJECT.parent
DATA_DIR = PROJECT / "data"
RESULTS_DIR = PROJECT / "results"

sys.path.insert(0, str(PROJECT))

from synthdoc.templates.registry import registry

templates = registry.list_templates()
print(f"{len(templates)} templates available:\n")
for i, t in enumerate(sorted(templates), 1):
    print(f"  {i:2}. {t}")


In [ ]:
TEMPLATES = None  # None = all registered templates
# TEMPLATES = [
#     "restaurant_receipt", "invoice", "business_card", "resume",
#     "bank_statement", "credit_card_statement", "irs_form_1040", "w2_form",
#     "passport", "driver_license",
#     "work_order", "aia_g702_g703", "bill_of_materials", "purchase_order",
#     "blueprint_technical_drawing",
#     "cms1500", "ub04_hospital_claims", "explanation_of_benefits",
#     "medical_prescription", "lab_report",
#     "closing_disclosure", "insurance_policy", "commercial_rent_roll",
#     "pay_stub", "remittance_advice",
#     "legal_pleading", "commercial_invoice", "certificate_of_analysis",
#     "home_inspection_report", "academic_transcript",
# ]

COUNT_PER_TEMPLATE = 1

In [ ]:
OCR_ENGINE = "glm"  # glm | tesseract | qwenvl | deepseek
NER_ENGINES = ["qwen3"]   # spacy | deberta | qwen3

## 2. Generate Dataset

In [ ]:
import subprocess

WORKERS = 16  # parallel template workers

cmd = [
    sys.executable, str(PROJECT / "generate_dataset.py"),
    "--count", str(COUNT_PER_TEMPLATE),
    "--workers", str(WORKERS),
]
if TEMPLATES:
    cmd += ["--templates"] + TEMPLATES

result = subprocess.run(cmd, cwd=str(PROJECT))
if result.returncode != 0:
    raise RuntimeError(f"Generation failed with exit code {result.returncode}")

In [ ]:
from synthdoc.bench.helpers import load_ner_pages, FULL_ENTITY_TYPES

pages = load_ner_pages(DATA_DIR, entity_types=FULL_ENTITY_TYPES)

print(f"Loaded {len(pages)} pages")
print(f"  Documents: {len(set(p['doc_id'] for p in pages))}")
print(f"  Templates: {len(set(p['template_type'] for p in pages))}")

## 3. OCR

In [ ]:
from synthdoc.bench.config import load_engines, _resolve, _dispatch_ocr
from synthdoc.bench.run_benchmark import _load_ocr_cache, _save_ocr_cache
from synthdoc.bench.helpers import compute_wer, compute_cer, InferenceTimer
import pandas as pd

engines_config = load_engines()
runner, model_id = _resolve(OCR_ENGINE, "ocr", engines_config)
ocr_key = f"{OCR_ENGINE}_ocr"

_load_ocr_cache(pages, ocr_key, DATA_DIR)
remaining = [p for p in pages if ocr_key not in p]

if remaining:
    print(f"Running {OCR_ENGINE} OCR on {len(remaining)}/{len(pages)} pages")
    with InferenceTimer(f"{OCR_ENGINE}_ocr") as timer:
        _dispatch_ocr(runner, remaining, model_id=model_id)
    _save_ocr_cache(pages, ocr_key, DATA_DIR)
    print(f"Done in {timer.result['time_s']}s")
else:
    print(f"All {len(pages)} pages already cached")

In [ ]:
rows = []
for p in pages:
    ocr_text = p.get(ocr_key, "")
    gt_text = p.get("gt_text", "")
    rows.append({
        "stem": p["stem"],
        "template": p["template_type"],
        "wer": compute_wer(gt_text, ocr_text),
        "cer": compute_cer(gt_text, ocr_text),
    })

ocr_df = pd.DataFrame(rows)
print(f"OCR Quality ({OCR_ENGINE}):")
print(f"  Mean WER: {ocr_df['wer'].mean():.4f}")
print(f"  Mean CER: {ocr_df['cer'].mean():.4f}")
print()
print(ocr_df.groupby("template")[["wer", "cer"]].mean().round(4).to_string())

## 4. NER

In [ ]:
import json
from synthdoc.bench.config import _dispatch_ner

NER_DIR = DATA_DIR / "ner"
NER_DIR.mkdir(exist_ok=True)

all_ner_results = {}

for ner_name in NER_ENGINES:
    model_label = f"{OCR_ENGINE}_{ner_name}"
    runner, model_id = _resolve(ner_name, "ner", engines_config)
    print(f"Running {model_label}...")

    with InferenceTimer(f"{model_label}_ner") as timer:
        ner_key = _dispatch_ner(runner, pages, ocr_key, model_id=model_id)

    # Save per-page NER results immediately after each engine finishes
    for p in pages:
        out_path = NER_DIR / f"{p['stem']}_{ner_name}.json"
        with open(out_path, "w") as f:
            json.dump({
                "stem": p["stem"],
                "template_type": p["template_type"],
                "ocr_engine": OCR_ENGINE,
                "ner_engine": ner_name,
                "ner_key": ner_key,
                "entities": p.get(ner_key, []),
            }, f, indent=2)

    all_ner_results[ner_name] = ner_key
    print(f"  Done in {timer.result['time_s']}s  (ner_key={ner_key})")
    print(f"  Saved {len(pages)} page results to {NER_DIR}/")

## 5. Evaluate

In [ ]:
from synthdoc.bench.run_benchmark import evaluate_model

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for ner_name in NER_ENGINES:
    ner_key = all_ner_results[ner_name]
    model_label = f"{OCR_ENGINE}_{ner_name}"
    results = evaluate_model(pages, model_label, ner_key, ocr_key)
    all_results.append(results)

    for eset in ["standard", "full"]:
        subset = results["ner_df"][results["ner_df"]["entity_set"] == eset]
        n = 7 if eset == "standard" else 14
        print(f"\n{model_label} [{eset} - {n} types]:")
        print(f"  Strict F1:  {subset['strict_f1'].mean():.4f}")
        print(f"  Relaxed F1: {subset['relaxed_f1'].mean():.4f}")

In [ ]:
combined = pd.concat([r["ner_df"] for r in all_results], ignore_index=True)

print("Summary (standard):")
std = combined[combined["entity_set"] == "standard"]
summary = std.groupby("ocr").agg({
    "strict_f1": "mean",
    "relaxed_f1": "mean",
    "strict_precision": "mean",
    "strict_recall": "mean",
}).round(4)
print(summary.to_string())

## 6. Save Results

In [ ]:
import json

combined.to_csv(RESULTS_DIR / "benchmark_results.csv", index=False)
if not ocr_df.empty:
    ocr_df.to_csv(RESULTS_DIR / "ocr_results.csv", index=False)

for r in all_results:
    model = r["model"]
    r["ner_df"].to_csv(RESULTS_DIR / f"benchmark_{model}.csv", index=False)
    r["type_df"].to_csv(RESULTS_DIR / f"per_type_{model}.csv", index=False)

print(f"Saved to {RESULTS_DIR}/")
for f in sorted(RESULTS_DIR.glob("*")):
    print(f"  {f.name}")

## 7. Spot-Check Failures

In [ ]:
from synthdoc.bench.helpers import detailed_match, NER_ENTITY_TYPES

ner_name = NER_ENGINES[0]
ner_key = all_ner_results[ner_name]
model_label = f"{OCR_ENGINE}_{ner_name}"

print(f"Missed entities for {model_label}:\n")

misses = []
for p in pages:
    gt = [e for e in p["gt_entities"] if e["type"] in NER_ENTITY_TYPES]
    pred = p.get(ner_key, [])
    _, unmatched_gt, _ = detailed_match(gt, pred)
    for etype, text in unmatched_gt:
        misses.append({"template": p["template_type"], "type": etype, "text": text})

if misses:
    miss_df = pd.DataFrame(misses)
    print(f"Total missed: {len(miss_df)}")
    print(f"\nBy type:")
    print(miss_df["type"].value_counts().to_string())
    print(f"\nBy template:")
    print(miss_df["template"].value_counts().to_string())
    print(f"\nExamples:")
    print(miss_df.head(20).to_string(index=False))
else:
    print("No misses - perfect recall!")